In [ ]:
import sys
import subprocess

print("Python usado pelo notebook:")
print(sys.executable)

print("\nInstalando...")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "scikit-posthocs"
])

print("\nTestando import...")
import scikit_posthocs as sp

print("OK!")
print("Versão:", sp.__version__)

Python usado pelo notebook:
/usr/bin/python3

Instalando...

Testando import...
OK!
Versão: 0.17.1


In [ ]:
import sys
!{sys.executable} -m pip install mrmr-selection

In [ ]:
import time
import pandas as pd
import numpy as np

from scipy import stats
import scikit_posthocs as sp
from mrmr import mrmr_classif

from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    cross_validate,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import random


In [ ]:
# =========================================================
# PARTE 1 - Carregamento e Isolamento de Amostras (Holdout 70/30)
# Ref.: Figura 1 do artigo - etapa "Isolamento de Amostras"
# =========================================================

# 1. Carrega o dataset
df = pd.read_csv('dataset_phishing.csv')  # ajuste o caminho se necessário

print('Shape original:', df.shape)
print(df['status'].value_counts())

# 2. Remove a coluna 'url' (identificador textual, nao e atributo preditivo)
df = df.drop(columns=['url'])

# 3. Codifica a variavel alvo: legitimate=0, phishing=1
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})

# 4. Separa features (X) e alvo (y)
X = df.drop(columns=['status'])
y = df['status']

# 5. Holdout 70% treino / 30% teste, estratificado para manter o balanceamento
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,          # mantem a proporcao 50/50 em treino e teste
    random_state=42       # reprodutibilidade
)

print('\nTreino:', X_train.shape, '| Teste:', X_test.shape)
print('\nProporcao treino:')
print(y_train.value_counts(normalize=True))
print('\nProporcao teste:')
print(y_test.value_counts(normalize=True))

Shape original: (11430, 89)
status
legitimate    5715
phishing      5715
Name: count, dtype: int64

Treino: (8001, 87) | Teste: (3429, 87)

Proporcao treino:
status
1    0.500062
0    0.499938
Name: proportion, dtype: float64

Proporcao teste:
status
0    0.500146
1    0.499854
Name: proportion, dtype: float64


In [ ]:
# =========================================================
# PARTE 2 - Pre-processamento: Normalizacao Z-Score
# Ref.: Equacao 1 do artigo -> z = (x - mu) / sigma
# Ajustado (fit) APENAS no treino, aplicado (transform) no treino e teste,
# para evitar data leakage.
# =========================================================

# --- (Parte 1, resumida) ---
df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# --- Parte 2: Normalizacao Z-Score ---
scaler = StandardScaler()

# fit_transform SOMENTE no treino (aprende media e desvio padrao do treino)
X_train_scaled = scaler.fit_transform(X_train)

# transform (sem fit) no teste, usando media/desvio do treino
X_test_scaled = scaler.transform(X_test)

# Reconverte para DataFrame, preservando nomes das colunas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print('Media apos normalizacao (treino, deve ser ~0):')
print(X_train_scaled.mean().abs().sort_values(ascending=False).head(3))

print('\nDesvio padrao apos normalizacao (treino, deve ser ~1):')
print(X_train_scaled.std().sort_values(ascending=False).head(3))

print('\nExemplo - antes e depois (coluna length_url, treino):')
print('Antes :', X_train['length_url'].head(3).values)
print('Depois:', X_train_scaled['length_url'].head(3).values)

print('\nShapes finais -> X_train_scaled:', X_train_scaled.shape, '| X_test_scaled:', X_test_scaled.shape)

Media apos normalizacao (treino, deve ser ~0):
nb_colon          2.557634e-16
nb_slash          1.869382e-16
ratio_intMedia    1.580760e-16
dtype: float64

Desvio padrao apos normalizacao (treino, deve ser ~1):
whois_registered_domain    1.000062
tld_in_path                1.000062
login_form                 1.000062
dtype: float64

Exemplo - antes e depois (coluna length_url, treino):
Antes : [17 80 50]
Depois: [-0.75881279  0.32814464 -0.18945413]

Shapes finais -> X_train_scaled: (8001, 87) | X_test_scaled: (3429, 87)


In [ ]:
# =========================================================
# PARTE 3 - Modelagem Comparativa (modelos scikit-learn)
# Ref.: Figura 1 - "Modelagem Comparativa" + "Validacao Cruzada Estratificada"
# Por enquanto usa TODOS os 87 atributos (sem CFIS ainda).
# Quando o CFIS chegar, e so trocar X_train_scaled/X_test_scaled
# pelo subconjunto filtrado e rodar de novo para comparar.
# =========================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Partes 1 e 2 (resumidas) ---
df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# --- Parte 3: modelos base + validacao cruzada estratificada 10-fold x 3 repeticoes ---
modelos = {
    'Regressao Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'Arvore de Decisao': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

resultados = []
for nome, modelo in modelos.items():
    scores = cross_validate(
        modelo, X_train_scaled, y_train,
        cv=cv,
        scoring=['f1', 'roc_auc'],
        n_jobs=-1
    )
    resultados.append({
        'Modelo': nome,
        'F1 media': scores['test_f1'].mean(),
        'F1 desvio': scores['test_f1'].std(),
        'AUC-ROC media': scores['test_roc_auc'].mean(),
        'AUC-ROC desvio': scores['test_roc_auc'].std(),
    })

resultados_df = pd.DataFrame(resultados).sort_values('F1 media', ascending=False)
print(resultados_df.to_string(index=False))

             Modelo  F1 media  F1 desvio  AUC-ROC media  AUC-ROC desvio
      Random Forest  0.964586   0.006504       0.992895        0.001674
Regressao Logistica  0.945637   0.007080       0.985535        0.002766
  Arvore de Decisao  0.931012   0.007960       0.930843        0.007857


In [ ]:
# =========================================================
# PARTE 3b - Modelagem Comparativa: MLP em PyTorch
# Ref.: Figura 1 - classificador "PyTorch MLP"
# Segue a mesma logica da Parte 3 (LogReg/DT/RF), mas com um
# Perceptron Multicamadas treinado via Adam + entropia cruzada binaria.
# =========================================================

# --- Partes 1 e 2 (resumidas) ---
df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # aqui fica como array numpy mesmo
X_test_scaled = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]


class MLP(nn.Module):
    """Perceptron Multicamadas simples: entrada -> 64 -> 32 -> 1 (logit)."""
    def __init__(self, n_inputs):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(n_inputs, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)   # saida = logit (sem sigmoid; usamos BCEWithLogitsLoss)
        )

    def forward(self, x):
        return self.rede(x)


def treinar_mlp(X_tr, y_tr, X_val, n_epochs=50, lr=1e-3):
    modelo = MLP(X_tr.shape[1]).to(device)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    perda_fn = nn.BCEWithLogitsLoss()

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr.values, dtype=torch.float32).view(-1, 1).to(device)
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)

    modelo.train()
    for epoca in range(n_epochs):
        otimizador.zero_grad()
        saida = modelo(X_tr_t)
        perda = perda_fn(saida, y_tr_t)
        perda.backward()
        otimizador.step()

    modelo.eval()
    with torch.no_grad():
        logits_val = modelo(X_val_t).cpu().numpy().ravel()
        probs_val = 1 / (1 + np.exp(-logits_val))  # sigmoid manual
    return probs_val


# --- Validacao cruzada estratificada 10-fold x 3 repeticoes ---
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

f1_scores, auc_scores = [], []
for fold_idx, (idx_tr, idx_val) in enumerate(cv.split(X_train_scaled, y_train)):
    X_tr, X_val = X_train_scaled[idx_tr], X_train_scaled[idx_val]
    y_tr, y_val = y_train.iloc[idx_tr], y_train.iloc[idx_val]

    probs_val = treinar_mlp(X_tr, y_tr, X_val)
    preds_val = (probs_val >= 0.5).astype(int)

    f1_scores.append(f1_score(y_val, preds_val))
    auc_scores.append(roc_auc_score(y_val, probs_val))

    if (fold_idx + 1) % 5 == 0:
        print(f'Fold {fold_idx + 1}/30 concluido')

print('\nMLP (PyTorch) - resultado da validacao cruzada:')
print(f'F1-Score  : media={np.mean(f1_scores):.4f}  desvio={np.std(f1_scores):.4f}')
print(f'AUC-ROC   : media={np.mean(auc_scores):.4f}  desvio={np.std(auc_scores):.4f}')

Fold 5/30 concluido
Fold 10/30 concluido
Fold 15/30 concluido
Fold 20/30 concluido
Fold 25/30 concluido
Fold 30/30 concluido

MLP (PyTorch) - resultado da validacao cruzada:
F1-Score  : media=0.9048  desvio=0.0127
AUC-ROC   : media=0.9663  desvio=0.0060


In [ ]:
# =========================================================
# PARTE 4 - Selecao de Atributos Hibrida (CFIS)
# Codigo original desenvolvido pelo Prof. Vinicius, adaptado aqui
# para rodar sobre o dataset_phishing.csv (em vez do breast-cancer.csv
# do notebook de pratica) e sobre os dados JA normalizados (Parte 2),
# calculado usando SOMENTE o conjunto de treino (sem vazamento de dados).
# =========================================================

# --- Partes 1 e 2 (resumidas) ---
df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)


# --- Parte 4: CFIS (codigo do professor, adaptado) ---
def combined_feature_selection(X, y):
    """
    Calcula o CFIS (Combined Feature Importance Score) para cada feature de X
    em relacao ao alvo y. Combina:
      1) Importancia por permutacao (normalizada, soma = 1)
      2) Coeficientes da Regressao Logistica (normalizados pela norma L2)
      3) Correlacao de Pearson com o alvo
    Retorna um DataFrame ordenado por CFIS (%) decrescente.
    """
    # Modelo base para extrair coeficientes e servir de estimador na permutacao
    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X, y)

    # 1) Importancia por permutacao
    result = permutation_importance(logreg, X, y, n_repeats=10, random_state=42, n_jobs=-1)
    importances_permutation = result.importances_mean
    normalized_importances_permutation = importances_permutation / np.sum(importances_permutation)

    # 2) Coeficientes normalizados (norma L2)
    coefficients = logreg.coef_[0]
    normalized_coefficients = coefficients / np.linalg.norm(coefficients)

    # 3) Correlacao de Pearson com o alvo
    correlation_scores = X.apply(lambda feature: feature.corr(y))

    # CFIS = media simples das 3 componentes (alpha=beta=gamma=1/3, Eq. 2 do artigo)
    combined_score = (normalized_importances_permutation + normalized_coefficients + correlation_scores) / 3
    cfis_percentage = (combined_score / np.sum(combined_score)) * 100

    feature_scores = pd.DataFrame({
        'Permutation Importance': normalized_importances_permutation,
        'Coefficient': coefficients,
        'Correlation': correlation_scores,
        'CFIS': combined_score,
        'CFIS (%)': cfis_percentage
    }, index=X.columns)

    return feature_scores.sort_values('CFIS (%)', ascending=False)


# Calcula CFIS usando SOMENTE o treino normalizado
cfis_scores = combined_feature_selection(X_train_scaled, y_train)

print('Top 15 atributos por CFIS:')
print(cfis_scores.head(15)[['CFIS (%)']])

# Seleciona o subconjunto otimizado (top 15, conforme estimado no artigo: 10-15 variaveis)
N_FEATURES = 15
top_features = cfis_scores.head(N_FEATURES).index.tolist()
print('\nAtributos selecionados:', top_features)

X_train_cfis = X_train_scaled[top_features]
X_test_cfis = X_test_scaled[top_features]

print('\nShape apos CFIS -> treino:', X_train_cfis.shape, '| teste:', X_test_cfis.shape)

cfis_scores.to_csv('cfis_scores.csv')

Top 15 atributos por CFIS:
                     CFIS (%)
google_index        19.742906
longest_words_raw   12.834157
phish_hints         11.276526
length_hostname      7.653307
domain_in_title      6.648817
ratio_digits_url     6.380800
ratio_digits_host    5.801669
nb_eq                5.690477
length_url           5.511590
nb_qm                5.479243
ip                   5.469552
nb_semicolumn        5.340042
shortest_word_host   4.678299
nb_slash             4.594716
tld_in_subdomain     4.241563

Atributos selecionados: ['google_index', 'longest_words_raw', 'phish_hints', 'length_hostname', 'domain_in_title', 'ratio_digits_url', 'ratio_digits_host', 'nb_eq', 'length_url', 'nb_qm', 'ip', 'nb_semicolumn', 'shortest_word_host', 'nb_slash', 'tld_in_subdomain']

Shape apos CFIS -> treino: (8001, 15) | teste: (3429, 15)


In [ ]:
# =========================================================
# PARTE 5 - Comparacao: 87 atributos (baseline) vs 15 atributos (CFIS)
# Ref.: Figura 1 completa ate "Avaliacao Estatistica"
# =========================================================


# --- Partes 1 e 2 ---
df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# --- Parte 4: CFIS (codigo do professor) ---
def combined_feature_selection(X, y):
    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X, y)
    result = permutation_importance(logreg, X, y, n_repeats=10, random_state=42, n_jobs=-1)
    normalized_importances_permutation = result.importances_mean / np.sum(result.importances_mean)
    coefficients = logreg.coef_[0]
    normalized_coefficients = coefficients / np.linalg.norm(coefficients)
    correlation_scores = X.apply(lambda feature: feature.corr(y))
    combined_score = (normalized_importances_permutation + normalized_coefficients + correlation_scores) / 3
    cfis_percentage = (combined_score / np.sum(combined_score)) * 100
    feature_scores = pd.DataFrame({
        'Permutation Importance': normalized_importances_permutation,
        'Coefficient': coefficients,
        'Correlation': correlation_scores,
        'CFIS': combined_score,
        'CFIS (%)': cfis_percentage
    }, index=X.columns)
    return feature_scores.sort_values('CFIS (%)', ascending=False)

cfis_scores = combined_feature_selection(X_train_scaled, y_train)
top_features = cfis_scores.head(15).index.tolist()
X_train_cfis = X_train_scaled[top_features]
X_test_cfis = X_test_scaled[top_features]

# --- Parte 5: roda os 3 modelos base nos dois cenarios ---
modelos = {
    'Regressao Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'Arvore de Decisao': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

cenarios = {
    'Baseline (87 atributos)': X_train_scaled,
    'CFIS (15 atributos)': X_train_cfis,
}

linhas = []
for nome_cenario, X_cv in cenarios.items():
    for nome_modelo, modelo in modelos.items():
        scores = cross_validate(
            modelo, X_cv, y_train, cv=cv,
            scoring=['f1', 'roc_auc'], n_jobs=-1
        )
        linhas.append({
            'Cenario': nome_cenario,
            'Modelo': nome_modelo,
            'N atributos': X_cv.shape[1],
            'F1 media': scores['test_f1'].mean(),
            'F1 desvio': scores['test_f1'].std(),
            'AUC-ROC media': scores['test_roc_auc'].mean(),
        })

resultados = pd.DataFrame(linhas)
pd.set_option('display.width', 120)
print(resultados.sort_values(['Modelo', 'Cenario']).to_string(index=False))

resultados.to_csv('comparacao_baseline_vs_cfis.csv', index=False)

                Cenario              Modelo  N atributos  F1 media  F1 desvio  AUC-ROC media
Baseline (87 atributos)   Arvore de Decisao           87  0.931012   0.007960       0.930843
    CFIS (15 atributos)   Arvore de Decisao           15  0.890897   0.009063       0.891158
Baseline (87 atributos)       Random Forest           87  0.964586   0.006504       0.992895
    CFIS (15 atributos)       Random Forest           15  0.924278   0.008637       0.973986
Baseline (87 atributos) Regressao Logistica           87  0.945637   0.007080       0.985535
    CFIS (15 atributos) Regressao Logistica           15  0.896165   0.009496       0.953830


In [ ]:
# =========================================================
# PARTE 5b - Curva: numero de atributos (CFIS) vs desempenho
# =========================================================


df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)

def combined_feature_selection(X, y):
    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X, y)
    result = permutation_importance(logreg, X, y, n_repeats=10, random_state=42, n_jobs=-1)
    normalized_importances_permutation = result.importances_mean / np.sum(result.importances_mean)
    coefficients = logreg.coef_[0]
    normalized_coefficients = coefficients / np.linalg.norm(coefficients)
    correlation_scores = X.apply(lambda feature: feature.corr(y))
    combined_score = (normalized_importances_permutation + normalized_coefficients + correlation_scores) / 3
    cfis_percentage = (combined_score / np.sum(combined_score)) * 100
    feature_scores = pd.DataFrame({'CFIS (%)': cfis_percentage}, index=X.columns)
    return feature_scores.sort_values('CFIS (%)', ascending=False)

cfis_scores = combined_feature_selection(X_train_scaled, y_train)
ranking = cfis_scores.index.tolist()

modelos = {
    'Regressao Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'Arvore de Decisao': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

n_values = [5, 10, 15, 20, 25, 30, 40, 50, 87]
linhas = []
for n in n_values:
    feats = ranking[:n]
    X_cv = X_train_scaled[feats]
    for nome_modelo, modelo in modelos.items():
        scores = cross_validate(modelo, X_cv, y_train, cv=cv, scoring=['f1', 'roc_auc'], n_jobs=-1)
        linhas.append({
            'N atributos': n,
            'Modelo': nome_modelo,
            'F1 media': scores['test_f1'].mean(),
            'AUC-ROC media': scores['test_roc_auc'].mean(),
        })
    print(f'n={n} concluido')

resultados = pd.DataFrame(linhas)
resultados.to_csv('curva_n_features.csv', index=False)

pivot_f1 = resultados.pivot(index='N atributos', columns='Modelo', values='F1 media')
print('\nF1-Score medio por numero de atributos:')
print(pivot_f1.round(4).to_string())

n=5 concluido
n=10 concluido
n=15 concluido
n=20 concluido
n=25 concluido
n=30 concluido
n=40 concluido
n=50 concluido
n=87 concluido

F1-Score medio por numero de atributos:
Modelo       Arvore de Decisao  Random Forest  Regressao Logistica
N atributos                                                       
5                       0.8794         0.8902               0.8861
10                      0.8830         0.9121               0.8953
15                      0.8909         0.9243               0.8962
20                      0.8919         0.9290               0.8980
25                      0.8944         0.9366               0.9060
30                      0.8939         0.9365               0.9097
40                      0.9108         0.9495               0.9104
50                      0.9084         0.9496               0.9133
87                      0.9318         0.9635               0.9456


In [ ]:
# =========================================================
# PARTE 5c - Custo computacional: tempo de treinamento
# por numero de atributos (CFIS) e por modelo
# =========================================================


df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)

# --- Tempo da propria etapa de selecao CFIS ---
def combined_feature_selection(X, y):
    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X, y)
    result = permutation_importance(logreg, X, y, n_repeats=10, random_state=42, n_jobs=-1)
    normalized_importances_permutation = result.importances_mean / np.sum(result.importances_mean)
    coefficients = logreg.coef_[0]
    normalized_coefficients = coefficients / np.linalg.norm(coefficients)
    correlation_scores = X.apply(lambda feature: feature.corr(y))
    combined_score = (normalized_importances_permutation + normalized_coefficients + correlation_scores) / 3
    cfis_percentage = (combined_score / np.sum(combined_score)) * 100
    feature_scores = pd.DataFrame({'CFIS (%)': cfis_percentage}, index=X.columns)
    return feature_scores.sort_values('CFIS (%)', ascending=False)

t0 = time.perf_counter()
cfis_scores = combined_feature_selection(X_train_scaled, y_train)
tempo_cfis = time.perf_counter() - t0
print(f'Tempo da etapa CFIS (calculo do ranking, 1x): {tempo_cfis:.3f}s\n')

ranking = cfis_scores.index.tolist()

modelos = {
    'Regressao Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'Arvore de Decisao': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

N_REPETICOES = 5  # repete o fit varias vezes p/ media estavel (evita ruido de 1 unica medicao)
n_values = [5, 15, 30, 87]

linhas = []
for n in n_values:
    feats = ranking[:n]
    X_n = X_train_scaled[feats]
    for nome_modelo, modelo_base in modelos.items():
        tempos = []
        for _ in range(N_REPETICOES):
            modelo = modelo_base.__class__(**modelo_base.get_params())
            t0 = time.perf_counter()
            modelo.fit(X_n, y_train)
            tempos.append(time.perf_counter() - t0)
        linhas.append({
            'N atributos': n,
            'Modelo': nome_modelo,
            'Tempo treino medio (s)': np.mean(tempos),
            'Tempo treino desvio (s)': np.std(tempos),
        })

tempos_df = pd.DataFrame(linhas)
pivot = tempos_df.pivot(index='N atributos', columns='Modelo', values='Tempo treino medio (s)')
print('Tempo medio de treinamento (segundos) por numero de atributos:')
print(pivot.round(4).to_string())

print('\nReducao percentual de tempo (15 atributos vs 87 - baseline usado na Parte 3):')
for modelo in pivot.columns:
    t87 = pivot.loc[87, modelo]
    t15 = pivot.loc[15, modelo]
    reducao = (1 - t15 / t87) * 100
    print(f'  {modelo}: {t15:.4f}s vs {t87:.4f}s -> reducao de {reducao:.1f}%')

print('\nReducao percentual de tempo (30 atributos vs 87):')
for modelo in pivot.columns:
    t87 = pivot.loc[87, modelo]
    t30 = pivot.loc[30, modelo]
    reducao = (1 - t30 / t87) * 100
    print(f'  {modelo}: {t30:.4f}s vs {t87:.4f}s -> reducao de {reducao:.1f}%')

tempos_df.to_csv('tempos_treinamento.csv', index=False)

Tempo da etapa CFIS (calculo do ranking, 1x): 4.755s

Tempo medio de treinamento (segundos) por numero de atributos:
Modelo       Arvore de Decisao  Random Forest  Regressao Logistica
N atributos                                                       
5                       0.0167         0.8131               0.0136
15                      0.0406         0.6880               0.0394
30                      0.0886         0.9335               0.0414
87                      0.2241         1.2882               0.7619

Reducao percentual de tempo (15 atributos vs 87 - baseline usado na Parte 3):
  Arvore de Decisao: 0.0406s vs 0.2241s -> reducao de 81.9%
  Random Forest: 0.6880s vs 1.2882s -> reducao de 46.6%
  Regressao Logistica: 0.0394s vs 0.7619s -> reducao de 94.8%

Reducao percentual de tempo (30 atributos vs 87):
  Arvore de Decisao: 0.0886s vs 0.2241s -> reducao de 60.4%
  Random Forest: 0.9335s vs 1.2882s -> reducao de 27.5%
  Regressao Logistica: 0.0414s vs 0.7619s -> reducao de 9

In [ ]:
# =========================================================
# PARTE 6 - Validacao estatistica: teste de Friedman + post-hoc de Nemenyi
# Ref.: secao 3 do artigo (Metodologia) - "os resultados serao submetidos
# ao teste nao parametrico de Friedman com o teste post-hoc de Nemenyi,
# adotando um nivel de significancia de alpha = 0.05"
#
# Compara os cenarios de selecao de atributos (5, 15, 30 e 87 - baseline)
# usando o Random Forest (modelo "estado da arte" citado no resumo).
# Os 30 scores de F1 por fold (10-fold x 3 repeticoes) sao PAREADOS entre
# os cenarios porque o mesmo RepeatedStratifiedKFold (mesma seed) e usado
# em todos eles -> exatamente o que o teste de Friedman exige (medidas
# repetidas no mesmo "sujeito", aqui cada fold).
# =========================================================

df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)

def combined_feature_selection(X, y):
    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X, y)
    result = permutation_importance(logreg, X, y, n_repeats=10, random_state=42, n_jobs=-1)
    normalized_importances_permutation = result.importances_mean / np.sum(result.importances_mean)
    coefficients = logreg.coef_[0]
    normalized_coefficients = coefficients / np.linalg.norm(coefficients)
    correlation_scores = X.apply(lambda feature: feature.corr(y))
    combined_score = (normalized_importances_permutation + normalized_coefficients + correlation_scores) / 3
    cfis_percentage = (combined_score / np.sum(combined_score)) * 100
    feature_scores = pd.DataFrame({'CFIS (%)': cfis_percentage}, index=X.columns)
    return feature_scores.sort_values('CFIS (%)', ascending=False)

ranking = combined_feature_selection(X_train_scaled, y_train).index.tolist()

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)
modelo = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

n_values = [5, 15, 30, 87]
scores_por_cenario = {}
for n in n_values:
    feats = ranking[:n]
    scores = cross_val_score(modelo, X_train_scaled[feats], y_train, cv=cv, scoring='f1', n_jobs=-1)
    scores_por_cenario[f'{n} atributos'] = scores
    print(f'{n} atributos: F1 medio = {scores.mean():.4f} (30 folds)')

# Matriz: 30 folds (linhas) x 4 cenarios (colunas) - dados PAREADOS
scores_df = pd.DataFrame(scores_por_cenario)
print('\nPrimeiras linhas da matriz de scores pareados (fold x cenario):')
print(scores_df.head())

# --- Teste de Friedman ---
stat, p_valor = stats.friedmanchisquare(*[scores_df[col] for col in scores_df.columns])
print(f'\nTeste de Friedman: estatistica={stat:.4f}, p-valor={p_valor:.6f}')

alpha = 0.05
if p_valor < alpha:
    print(f'p-valor < {alpha} -> ha diferenca estatisticamente significativa entre pelo menos dois cenarios.')
    print('Prosseguindo com post-hoc de Nemenyi...\n')

    # --- Post-hoc de Nemenyi ---
    nemenyi = sp.posthoc_nemenyi_friedman(scores_df.values)
    nemenyi.columns = scores_df.columns
    nemenyi.index = scores_df.columns
    print('Matriz de p-valores do teste de Nemenyi (par a par):')
    print(nemenyi.round(4).to_string())

    print('\nInterpretacao (alpha = 0.05):')
    for i, c1 in enumerate(scores_df.columns):
        for c2 in scores_df.columns[i+1:]:
            p = nemenyi.loc[c1, c2]
            resultado = 'DIFERENCA SIGNIFICATIVA' if p < alpha else 'sem diferenca significativa'
            print(f'  {c1} vs {c2}: p={p:.4f} -> {resultado}')
else:
    print(f'p-valor >= {alpha} -> nao ha evidencia de diferenca entre os cenarios (Friedman nao rejeitou H0).')

scores_df.to_csv('friedman_scores_por_fold.csv', index=False)

5 atributos: F1 medio = 0.8902 (30 folds)
15 atributos: F1 medio = 0.9243 (30 folds)
30 atributos: F1 medio = 0.9365 (30 folds)
87 atributos: F1 medio = 0.9635 (30 folds)

Primeiras linhas da matriz de scores pareados (fold x cenario):
   5 atributos  15 atributos  30 atributos  87 atributos
0     0.905055      0.935162      0.941323      0.967581
1     0.860697      0.901619      0.911392      0.957179
2     0.891626      0.929193      0.927250      0.955224
3     0.892500      0.921519      0.939241      0.959494
4     0.900878      0.945137      0.957179      0.969773

Teste de Friedman: estatistica=88.8400, p-valor=0.000000
p-valor < 0.05 -> ha diferenca estatisticamente significativa entre pelo menos dois cenarios.
Prosseguindo com post-hoc de Nemenyi...

Matriz de p-valores do teste de Nemenyi (par a par):
              5 atributos  15 atributos  30 atributos  87 atributos
5 atributos        1.0000        0.0104        0.0000        0.0000
15 atributos       0.0104        1.0000 

In [ ]:
# =========================================================
# PARTE 7 - Comparacao CFIS vs metodos tradicionais (LASSO, mRMR, CFS)
# Ref.: Metodologia do artigo - "os resultados serao submetidos ao teste
# nao parametrico de Friedman ... entre os subconjuntos de atributos
# gerados pelo CFIS e pelos metodos tradicionais (LASSO, mRMR, CFS)"
# =========================================================


N_FEATURES = 15  # mesmo N usado na Parte 4, para comparacao justa

df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)


# --- 1) CFIS (codigo do professor) ---
def select_cfis(X, y, n):
    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X, y)
    result = permutation_importance(logreg, X, y, n_repeats=10, random_state=42, n_jobs=-1)
    normalized_importances_permutation = result.importances_mean / np.sum(result.importances_mean)
    coefficients = logreg.coef_[0]
    normalized_coefficients = coefficients / np.linalg.norm(coefficients)
    correlation_scores = X.apply(lambda feature: feature.corr(y))
    combined_score = (normalized_importances_permutation + normalized_coefficients + correlation_scores) / 3
    cfis_percentage = (combined_score / np.sum(combined_score)) * 100
    ranking = pd.Series(cfis_percentage, index=X.columns).sort_values(ascending=False)
    return ranking.head(n).index.tolist()


# --- 2) LASSO (Regressao Logistica com penalidade L1) ---
def select_lasso(X, y, n):
    lasso = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42)
    lasso.fit(X, y)
    coef_abs = pd.Series(np.abs(lasso.coef_[0]), index=X.columns)
    return coef_abs.sort_values(ascending=False).head(n).index.tolist()


# --- 3) mRMR (Max-Relevance Min-Redundancy, biblioteca mrmr_selection) ---
def select_mrmr(X, y, n):
    selecionadas = mrmr_classif(X=X, y=y, K=n, show_progress=False)
    return selecionadas


# --- 4) CFS (Correlation-based Feature Selection, algoritmo de Hall 1999) ---
def select_cfs(X, y, n):
    """
    Busca gulosa (forward) maximizando o Merit:
        Merit_s = k * r_cf_media / sqrt(k + k*(k-1)*r_ff_media)
    onde r_cf_media = correlacao media feature-alvo das features selecionadas,
    r_ff_media = correlacao media entre pares de features selecionadas.
    Executa ate atingir n features (para permitir comparacao justa com os
    demais metodos, que tambem selecionam um N fixo).
    """
    corr_target = X.apply(lambda col: abs(col.corr(y)))
    corr_matrix = X.corr().abs()

    selecionadas = []
    restantes = list(X.columns)

    while len(selecionadas) < n and restantes:
        melhor_merito = -np.inf
        melhor_feat = None
        for feat in restantes:
            candidato = selecionadas + [feat]
            k = len(candidato)
            r_cf = corr_target[candidato].mean()
            if k > 1:
                pares = corr_matrix.loc[candidato, candidato].values
                # media das correlacoes fora da diagonal
                soma_off_diag = pares.sum() - np.trace(pares)
                r_ff = soma_off_diag / (k * (k - 1))
            else:
                r_ff = 0.0
            denominador = np.sqrt(k + k * (k - 1) * r_ff)
            merito = (k * r_cf) / denominador if denominador > 0 else 0.0
            if merito > melhor_merito:
                melhor_merito = merito
                melhor_feat = feat
        selecionadas.append(melhor_feat)
        restantes.remove(melhor_feat)

    return selecionadas


print(f'Selecionando {N_FEATURES} atributos por cada metodo...\n')
feats_cfis = select_cfis(X_train_scaled, y_train, N_FEATURES)
print('CFIS:', feats_cfis)
feats_lasso = select_lasso(X_train_scaled, y_train, N_FEATURES)
print('LASSO:', feats_lasso)
feats_mrmr = select_mrmr(X_train_scaled, y_train, N_FEATURES)
print('mRMR:', feats_mrmr)
feats_cfs = select_cfs(X_train_scaled, y_train, N_FEATURES)
print('CFS:', feats_cfs)

metodos = {
    'CFIS': feats_cfis,
    'LASSO': feats_lasso,
    'mRMR': feats_mrmr,
    'CFS': feats_cfs,
    'Baseline (87)': X_train_scaled.columns.tolist(),
}

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)
modelo = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

scores_por_metodo = {}
print('\nDesempenho (Random Forest, F1) por metodo de selecao:')
for nome, feats in metodos.items():
    scores = cross_val_score(modelo, X_train_scaled[feats], y_train, cv=cv, scoring='f1', n_jobs=-1)
    scores_por_metodo[nome] = scores
    print(f'  {nome}: F1 medio = {scores.mean():.4f} (desvio {scores.std():.4f})')

scores_df = pd.DataFrame(scores_por_metodo)
scores_df.to_csv('friedman_metodos_selecao.csv', index=False)

# --- Friedman + Nemenyi entre os metodos ---
stat, p_valor = stats.friedmanchisquare(*[scores_df[col] for col in scores_df.columns])
print(f'\nTeste de Friedman (entre metodos): estatistica={stat:.4f}, p-valor={p_valor:.6f}')

alpha = 0.05
if p_valor < alpha:
    print(f'p-valor < {alpha} -> ha diferenca estatisticamente significativa entre pelo menos dois metodos.\n')
    nemenyi = sp.posthoc_nemenyi_friedman(scores_df.values)
    nemenyi.columns = scores_df.columns
    nemenyi.index = scores_df.columns
    print('Matriz de p-valores do teste de Nemenyi (par a par):')
    print(nemenyi.round(4).to_string())

    print('\nComparacoes envolvendo o CFIS (alpha = 0.05):')
    for outro in scores_df.columns:
        if outro == 'CFIS':
            continue
        p = nemenyi.loc['CFIS', outro]
        resultado = 'DIFERENCA SIGNIFICATIVA' if p < alpha else 'SEM diferenca significativa'
        print(f'  CFIS vs {outro}: p={p:.4f} -> {resultado}')
else:
    print(f'p-valor >= {alpha} -> nao ha evidencia de diferenca entre os metodos.')

Selecionando 15 atributos por cada metodo...

CFIS: ['google_index', 'longest_words_raw', 'phish_hints', 'length_hostname', 'domain_in_title', 'ratio_digits_url', 'ratio_digits_host', 'nb_eq', 'length_url', 'nb_qm', 'ip', 'nb_semicolumn', 'shortest_word_host', 'nb_slash', 'tld_in_subdomain']
LASSO: ['page_rank', 'google_index', 'longest_words_raw', 'phish_hints', 'nb_www', 'nb_hyperlinks', 'nb_hyphens', 'length_hostname', 'nb_semicolumn', 'ratio_digits_host', 'avg_word_host', 'nb_qm', 'nb_underscore', 'ratio_extMedia', 'domain_age']
mRMR: ['google_index', 'page_rank', 'nb_www', 'nb_hyperlinks', 'phish_hints', 'ratio_digits_url', 'domain_in_title', 'nb_qm', 'domain_age', 'ratio_intHyperlinks', 'length_hostname', 'ip', 'domain_with_copyright', 'nb_slash', 'ratio_digits_host']
CFS: ['google_index', 'page_rank', 'nb_www', 'nb_qm', 'nb_hyperlinks', 'phish_hints', 'ratio_intHyperlinks', 'ratio_digits_url', 'domain_age', 'length_hostname', 'domain_in_title', 'domain_with_copyright', 'avg_word

In [ ]:
# =========================================================
# PARTE 8 - Variante do CFIS com |correlacao| (Eq. 2 do artigo)
# Compara: CFIS original (codigo do professor) vs CFIS com |rho|
# vs LASSO, mRMR, CFS e baseline. N = 15 atributos, Random Forest.
# =========================================================

N_FEATURES = 15

df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)


def select_cfis(X, y, n, usar_abs=False):
    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X, y)
    result = permutation_importance(logreg, X, y, n_repeats=10, random_state=42, n_jobs=-1)
    perm = result.importances_mean / np.sum(result.importances_mean)
    coefficients = logreg.coef_[0]
    coef = coefficients / np.linalg.norm(coefficients)
    corr = X.apply(lambda feature: feature.corr(y))
    if usar_abs:
        coef = np.abs(coef)
        corr = corr.abs()
    combined = (perm + coef + corr) / 3
    ranking = pd.Series(combined, index=X.columns).sort_values(ascending=False)
    return ranking.head(n).index.tolist()


def select_lasso(X, y, n):
    lasso = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42)
    lasso.fit(X, y)
    return pd.Series(np.abs(lasso.coef_[0]), index=X.columns).sort_values(ascending=False).head(n).index.tolist()


def select_mrmr(X, y, n):
    return mrmr_classif(X=X, y=y, K=n, show_progress=False)


def select_cfs(X, y, n):
    corr_target = X.apply(lambda col: abs(col.corr(y)))
    corr_matrix = X.corr().abs()
    sel, rest = [], list(X.columns)
    while len(sel) < n and rest:
        melhor, feat_m = -np.inf, None
        for f in rest:
            cand = sel + [f]
            k = len(cand)
            r_cf = corr_target[cand].mean()
            if k > 1:
                p = corr_matrix.loc[cand, cand].values
                r_ff = (p.sum() - np.trace(p)) / (k * (k - 1))
            else:
                r_ff = 0.0
            den = np.sqrt(k + k * (k - 1) * r_ff)
            m = (k * r_cf) / den if den > 0 else 0.0
            if m > melhor:
                melhor, feat_m = m, f
        sel.append(feat_m)
        rest.remove(feat_m)
    return sel


metodos = {
    'CFIS (original)': select_cfis(X_train_scaled, y_train, N_FEATURES, usar_abs=False),
    'CFIS (|rho|)': select_cfis(X_train_scaled, y_train, N_FEATURES, usar_abs=True),
    'LASSO': select_lasso(X_train_scaled, y_train, N_FEATURES),
    'mRMR': select_mrmr(X_train_scaled, y_train, N_FEATURES),
    'CFS': select_cfs(X_train_scaled, y_train, N_FEATURES),
    'Baseline (87)': X_train_scaled.columns.tolist(),
}

print('Atributos CFIS (|rho|):', metodos['CFIS (|rho|)'])
print('\nAtributos em comum entre CFIS original e CFIS (|rho|):',
      len(set(metodos['CFIS (original)']) & set(metodos['CFIS (|rho|)'])), 'de', N_FEATURES)

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)
modelo = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

scores = {}
print('\nF1 (Random Forest, 30 folds):')
for nome, feats in metodos.items():
    s = cross_val_score(modelo, X_train_scaled[feats], y_train, cv=cv, scoring='f1', n_jobs=-1)
    scores[nome] = s
    print(f'  {nome}: {s.mean():.4f} (desvio {s.std():.4f})')

scores_df = pd.DataFrame(scores)
scores_df.to_csv('friedman_cfis_abs.csv', index=False)

stat, p = stats.friedmanchisquare(*[scores_df[c] for c in scores_df.columns])
print(f'\nFriedman: estatistica={stat:.4f}, p={p:.6f}')
nem = sp.posthoc_nemenyi_friedman(scores_df.values)
nem.columns = scores_df.columns
nem.index = scores_df.columns
print('\nNemenyi (p-valores):')
print(nem.round(4).to_string())
print('\nCFIS (|rho|) vs demais (alpha=0.05):')
for outro in scores_df.columns:
    if outro == 'CFIS (|rho|)':
        continue
    pv = nem.loc['CFIS (|rho|)', outro]
    print(f'  vs {outro}: p={pv:.4f} -> {"DIFERENTE" if pv < 0.05 else "sem diferenca significativa"}')

Atributos CFIS (|rho|): ['google_index', 'page_rank', 'longest_words_raw', 'nb_www', 'phish_hints', 'nb_hyperlinks', 'length_hostname', 'avg_word_host', 'domain_age', 'domain_in_title', 'ratio_digits_url', 'nb_hyphens', 'ratio_digits_host', 'nb_eq', 'length_url']

Atributos em comum entre CFIS original e CFIS (|rho|): 9 de 15

F1 (Random Forest, 30 folds):
  CFIS (original): 0.9243 (desvio 0.0086)
  CFIS (|rho|): 0.9607 (desvio 0.0059)
  LASSO: 0.9605 (desvio 0.0059)
  mRMR: 0.9594 (desvio 0.0052)
  CFS: 0.9592 (desvio 0.0064)
  Baseline (87): 0.9646 (desvio 0.0065)

Friedman: estatistica=79.9185, p=0.000000

Nemenyi (p-valores):
                 CFIS (original)  CFIS (|rho|)   LASSO    mRMR     CFS  Baseline (87)
CFIS (original)              1.0        0.0000  0.0000  0.0000  0.0000         0.0000
CFIS (|rho|)                 0.0        1.0000  1.0000  0.9996  0.9790         0.0640
LASSO                        0.0        1.0000  1.0000  0.9968  0.9473         0.1002
mRMR              

In [ ]:
# =========================================================
# PARTE 9 - MLP (PyTorch): Baseline (87) vs CFIS com |rho| (15)
# Rodar no Google Colab (PyTorch ja disponivel).
# Antes de rodar: faca upload do dataset_phishing.csv no Colab
# (icone de pasta na barra lateral -> upload).
# Saida: F1 e AUC-ROC por fold (10-fold x 3 repeticoes) para cada cenario,
# salvos em mlp_scores_por_fold.csv (pronto para Friedman/Nemenyi depois).
# =========================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

N_FEATURES = 15
N_EPOCHS = 30
BATCH_SIZE = 256
LR = 1e-3

# --- Partes 1 e 2: carga, holdout 70/30 e Z-score (fit so no treino) ---
df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                              columns=X_train.columns, index=X_train.index)


# --- Parte 8: CFIS com |rho| (variante alinhada a Equacao 2 do artigo) ---
def select_cfis_abs(X, y, n):
    logreg = LogisticRegression(max_iter=1000, random_state=SEED)
    logreg.fit(X, y)
    result = permutation_importance(logreg, X, y, n_repeats=10,
                                    random_state=SEED, n_jobs=-1)
    perm = result.importances_mean / np.sum(result.importances_mean)
    coefficients = logreg.coef_[0]
    coef = np.abs(coefficients / np.linalg.norm(coefficients))
    corr = X.apply(lambda feature: feature.corr(y)).abs()
    combined = (perm + coef + corr) / 3
    ranking = pd.Series(combined, index=X.columns).sort_values(ascending=False)
    return ranking.head(n).index.tolist()


feats_cfis = select_cfis_abs(X_train_scaled, y_train, N_FEATURES)
print('Atributos CFIS (|rho|):', feats_cfis)


# --- MLP: entrada -> 64 -> 32 -> 1 (logit) ---
class MLP(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(n_inputs, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.rede(x)


def treinar_e_prever(X_tr, y_tr, X_val):
    modelo = MLP(X_tr.shape[1]).to(device)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=LR)
    perda_fn = nn.BCEWithLogitsLoss()

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32, device=device).view(-1, 1)
    X_val_t = torch.tensor(X_val, dtype=torch.float32, device=device)

    n = X_tr_t.shape[0]
    modelo.train()
    for _ in range(N_EPOCHS):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            otimizador.zero_grad()
            perda = perda_fn(modelo(X_tr_t[idx]), y_tr_t[idx])
            perda.backward()
            otimizador.step()

    modelo.eval()
    with torch.no_grad():
        logits = modelo(X_val_t).cpu().numpy().ravel()
    return 1 / (1 + np.exp(-logits))


# --- Validacao cruzada estratificada 10-fold x 3 repeticoes ---
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=SEED)

cenarios = {
    'Baseline (87)': X_train_scaled.columns.tolist(),
    'CFIS |rho| (15)': feats_cfis,
}

resultados = {}
for nome, feats in cenarios.items():
    X_arr = X_train_scaled[feats].values
    y_arr = y_train.values
    f1s, aucs = [], []
    for fold_idx, (i_tr, i_val) in enumerate(cv.split(X_arr, y_arr), start=1):
        probs = treinar_e_prever(X_arr[i_tr], y_arr[i_tr], X_arr[i_val])
        preds = (probs >= 0.5).astype(int)
        f1s.append(f1_score(y_arr[i_val], preds))
        aucs.append(roc_auc_score(y_arr[i_val], probs))
        if fold_idx % 10 == 0:
            print(f'{nome}: fold {fold_idx}/30 concluido')
    resultados[f'{nome} - F1'] = f1s
    resultados[f'{nome} - AUC'] = aucs

res_df = pd.DataFrame(resultados)
res_df.to_csv('mlp_scores_por_fold.csv', index=False)

print('\nMLP (PyTorch) - resultado da validacao cruzada (30 folds):')
for nome in cenarios:
    f1 = res_df[f'{nome} - F1']
    auc = res_df[f'{nome} - AUC']
    print(f'{nome}: F1 = {f1.mean():.4f} (desvio {f1.std():.4f}) | '
          f'AUC-ROC = {auc.mean():.4f} (desvio {auc.std():.4f})')

Dispositivo: cpu
Atributos CFIS (|rho|): ['google_index', 'page_rank', 'longest_words_raw', 'nb_www', 'phish_hints', 'nb_hyperlinks', 'length_hostname', 'avg_word_host', 'domain_age', 'domain_in_title', 'ratio_digits_url', 'nb_hyphens', 'ratio_digits_host', 'nb_eq', 'length_url']
Baseline (87): fold 10/30 concluido
Baseline (87): fold 20/30 concluido
Baseline (87): fold 30/30 concluido
CFIS |rho| (15): fold 10/30 concluido
CFIS |rho| (15): fold 20/30 concluido
CFIS |rho| (15): fold 30/30 concluido

MLP (PyTorch) - resultado da validacao cruzada (30 folds):
Baseline (87): F1 = 0.9569 (desvio 0.0072) | AUC-ROC = 0.9893 (desvio 0.0032)
CFIS |rho| (15): F1 = 0.9494 (desvio 0.0071) | AUC-ROC = 0.9878 (desvio 0.0022)


In [ ]:
# =========================================================
# PARTE 10 - Avaliacao final no conjunto de TESTE (30%)
# Ref.: Figura 1 - "Avaliacao Estatistica / Teste final de generalizacao"
#
# Regras para evitar vazamento de dados:
#  - scaler e selecao CFIS (|rho|) ajustados SO no treino
#  - cada modelo e treinado UMA vez no treino completo (70%)
#  - o teste (30%) e usado UMA vez, apenas para reportar as metricas
#  - nenhum hiperparametro foi ajustado olhando o teste
#
# Os modelos scikit-learn rodam em qualquer lugar. O MLP so roda se o
# PyTorch estiver instalado (no Colab ja esta).
# =========================================================
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
N_FEATURES = 15

# --- Partes 1 e 2 ---
df = pd.read_csv('dataset_phishing.csv')
df = df.drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train),
                         columns=X_train.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test),
                        columns=X_test.columns, index=X_test.index)


# --- CFIS com |rho| (selecao feita SOMENTE com o treino) ---
def select_cfis_abs(X, y, n):
    logreg = LogisticRegression(max_iter=1000, random_state=SEED)
    logreg.fit(X, y)
    result = permutation_importance(logreg, X, y, n_repeats=10,
                                    random_state=SEED, n_jobs=-1)
    perm = result.importances_mean / np.sum(result.importances_mean)
    coefficients = logreg.coef_[0]
    coef = np.abs(coefficients / np.linalg.norm(coefficients))
    corr = X.apply(lambda feature: feature.corr(y)).abs()
    combined = (perm + coef + corr) / 3
    ranking = pd.Series(combined, index=X.columns).sort_values(ascending=False)
    return ranking.head(n).index.tolist()


feats_cfis = select_cfis_abs(X_train_s, y_train, N_FEATURES)
cenarios = {
    'Baseline (87)': X_train_s.columns.tolist(),
    'CFIS |rho| (15)': feats_cfis,
}

modelos = {
    'Regressao Logistica': lambda: LogisticRegression(max_iter=1000, random_state=SEED),
    'Arvore de Decisao': lambda: DecisionTreeClassifier(random_state=SEED),
    'Random Forest': lambda: RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
}

linhas = []


def registrar(cenario, modelo_nome, y_true, probs):
    preds = (probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    linhas.append({
        'Cenario': cenario, 'Modelo': modelo_nome,
        'F1': f1_score(y_true, preds),
        'AUC-ROC': roc_auc_score(y_true, probs),
        'VP': tp, 'FP': fp, 'VN': tn, 'FN': fn,
    })


# --- Modelos scikit-learn ---
for cenario, feats in cenarios.items():
    for nome, fabrica in modelos.items():
        m = fabrica()
        m.fit(X_train_s[feats], y_train)
        probs = m.predict_proba(X_test_s[feats])[:, 1]
        registrar(cenario, nome, y_test.values, probs)

# --- MLP (PyTorch) - roda apenas se o PyTorch estiver disponivel ---
try:
    import torch
    import torch.nn as nn

    torch.manual_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    class MLP(nn.Module):
        def __init__(self, n_inputs):
            super().__init__()
            self.rede = nn.Sequential(
                nn.Linear(n_inputs, 64), nn.ReLU(),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, 1),
            )

        def forward(self, x):
            return self.rede(x)

    def treinar_e_prever(X_tr, y_tr, X_te, n_epochs=30, batch=256, lr=1e-3):
        modelo = MLP(X_tr.shape[1]).to(device)
        otim = torch.optim.Adam(modelo.parameters(), lr=lr)
        perda_fn = nn.BCEWithLogitsLoss()
        Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)
        yt = torch.tensor(y_tr, dtype=torch.float32, device=device).view(-1, 1)
        Xe = torch.tensor(X_te, dtype=torch.float32, device=device)
        n = Xt.shape[0]
        modelo.train()
        for _ in range(n_epochs):
            perm = torch.randperm(n, device=device)
            for i in range(0, n, batch):
                idx = perm[i:i + batch]
                otim.zero_grad()
                perda_fn(modelo(Xt[idx]), yt[idx]).backward()
                otim.step()
        modelo.eval()
        with torch.no_grad():
            logits = modelo(Xe).cpu().numpy().ravel()
        return 1 / (1 + np.exp(-logits))

    for cenario, feats in cenarios.items():
        probs = treinar_e_prever(X_train_s[feats].values, y_train.values,
                                 X_test_s[feats].values)
        registrar(cenario, 'MLP (PyTorch)', y_test.values, probs)
except ImportError:
    print('PyTorch nao encontrado: MLP ignorado (rode no Colab para inclui-lo).')

res = pd.DataFrame(linhas).sort_values(['Modelo', 'Cenario'])
pd.set_option('display.width', 140)
print(res.round(4).to_string(index=False))
res.to_csv('avaliacao_teste.csv', index=False)

        Cenario              Modelo     F1  AUC-ROC   VP  FP   VN  FN
  Baseline (87)   Arvore de Decisao 0.9384   0.9379 1623 122 1593  91
CFIS |rho| (15)   Arvore de Decisao 0.9379   0.9376 1615 115 1600  99
  Baseline (87)       MLP (PyTorch) 0.9607   0.9908 1652  73 1642  62
CFIS |rho| (15)       MLP (PyTorch) 0.9473   0.9884 1618  84 1631  96
  Baseline (87)       Random Forest 0.9646   0.9930 1661  69 1646  53
CFIS |rho| (15)       Random Forest 0.9574   0.9913 1651  84 1631  63
  Baseline (87) Regressao Logistica 0.9395   0.9842 1607 100 1615 107
CFIS |rho| (15) Regressao Logistica 0.9346   0.9787 1614 126 1589 100


In [ ]:
import time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from mrmr import mrmr_classif

SEED, N = 42, 15
df = pd.read_csv('dataset_phishing.csv').drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})
X, y = df.drop(columns=['status']), df['status']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
sc = StandardScaler()
Xs = pd.DataFrame(sc.fit_transform(Xtr), columns=Xtr.columns, index=Xtr.index)

def cfis_abs(X, y, n):
    lr = LogisticRegression(max_iter=1000, random_state=SEED).fit(X, y)
    r = permutation_importance(lr, X, y, n_repeats=10, random_state=SEED, n_jobs=-1)
    perm = r.importances_mean / r.importances_mean.sum()
    c = lr.coef_[0]; coef = np.abs(c / np.linalg.norm(c))
    corr = X.apply(lambda f: f.corr(y)).abs()
    return pd.Series((perm + coef + corr) / 3, index=X.columns).sort_values(ascending=False).head(n).index.tolist()

def lasso(X, y, n):
    m = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=SEED).fit(X, y)
    return pd.Series(np.abs(m.coef_[0]), index=X.columns).sort_values(ascending=False).head(n).index.tolist()

def mrmr(X, y, n):
    return mrmr_classif(X=X, y=y, K=n, show_progress=False)

def cfs(X, y, n):
    ct = X.apply(lambda col: abs(col.corr(y))); cm = X.corr().abs()
    sel, rest = [], list(X.columns)
    while len(sel) < n and rest:
        best, bf = -np.inf, None
        for f in rest:
            cand = sel + [f]; k = len(cand); rcf = ct[cand].mean()
            if k > 1:
                p = cm.loc[cand, cand].values; rff = (p.sum() - np.trace(p)) / (k * (k - 1))
            else: rff = 0.0
            den = np.sqrt(k + k * (k - 1) * rff); m = k * rcf / den if den > 0 else 0
            if m > best: best, bf = m, f
        sel.append(bf); rest.remove(bf)
    return sel

# tempos de selecao (3 execucoes cada)
print('TEMPO DE SELECAO (s, media de 3 execucoes):')
tempos_sel = {}
for nome, fn in [('CFIS', cfis_abs), ('LASSO', lasso), ('mRMR', mrmr), ('CFS', cfs)]:
    ts = []
    for _ in range(3):
        t0 = time.perf_counter(); fn(Xs, ytr, N); ts.append(time.perf_counter() - t0)
    tempos_sel[nome] = (np.mean(ts), np.std(ts))
    print(f'  {nome}: {np.mean(ts):.3f} +- {np.std(ts):.3f}')
pd.DataFrame(tempos_sel, index=['media', 'desvio']).T.to_csv('tempos_selecao.csv')

feats = cfis_abs(Xs, ytr, N)
cen = {'Baseline (87)': Xs.columns.tolist(), 'CFIS |rho| (15)': feats}
mods = {
    'Regressao Logistica': lambda: LogisticRegression(max_iter=1000, random_state=SEED),
    'Arvore de Decisao': lambda: DecisionTreeClassifier(random_state=SEED),
    'Random Forest': lambda: RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
}
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=SEED)
linhas = []
print('\nVALIDACAO CRUZADA (30 folds) e TEMPO DE TREINO:')
for cn, cols in cen.items():
    for mn, fab in mods.items():
        s = cross_validate(fab(), Xs[cols], ytr, cv=cv, scoring=['f1', 'roc_auc'], n_jobs=-1)
        ts = []
        for _ in range(5):
            m = fab(); t0 = time.perf_counter(); m.fit(Xs[cols], ytr); ts.append(time.perf_counter() - t0)
        linhas.append(dict(Cenario=cn, Modelo=mn, F1=s['test_f1'].mean(), F1_sd=s['test_f1'].std(),
                           AUC=s['test_roc_auc'].mean(), AUC_sd=s['test_roc_auc'].std(), Tempo_s=np.mean(ts)))
out = pd.DataFrame(linhas)
out.to_csv('cv_e_tempo_final.csv', index=False)
print(out.round(4).to_string(index=False))

TEMPO DE SELECAO (s, media de 3 execucoes):
  CFIS: 14.842 +- 8.189
  LASSO: 0.136 +- 0.007
  mRMR: 2.273 +- 0.363
  CFS: 1.403 +- 0.024

VALIDACAO CRUZADA (30 folds) e TEMPO DE TREINO:
        Cenario              Modelo     F1  F1_sd    AUC  AUC_sd  Tempo_s
  Baseline (87) Regressao Logistica 0.9456 0.0071 0.9855  0.0028   0.3589
  Baseline (87)   Arvore de Decisao 0.9310 0.0080 0.9308  0.0079   0.2186
  Baseline (87)       Random Forest 0.9646 0.0065 0.9929  0.0017   1.4153
CFIS |rho| (15) Regressao Logistica 0.9341 0.0085 0.9792  0.0037   0.0233
CFIS |rho| (15)   Arvore de Decisao 0.9331 0.0073 0.9330  0.0075   0.0579
CFIS |rho| (15)       Random Forest 0.9607 0.0059 0.9908  0.0020   1.0248
